# Python and CLI for Data Science - Session 07

- *Course*: Python and CLI for Data Science
- *Session*: 07
- *Unit*: Pandas Hierarchical Indexing

### (Re)sources:
- [Python Data Science Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/index.html) _by Jake VanderPlas (Code released under the MIT License)_
- [Pandas Cookbook](https://github.com/jvns/pandas-cookbook/tree/master) _by Julia Evans (Code released under the CC BY-SA 4.0 DEED License)_

In [ ]:
import numpy as np
import pandas as pd

# Hierarchical Indexing

- Up to this point we've been focused primarily on one-dimensional (`Series`) and two-dimensional (`DataFrame`) data
- Pandas also supports higher-dimensional data using *hierarchical indexing* (also known as *multi-indexing*)

## A Multiply Indexed Series

- Let's start by considering how we might represent two-dimensional data within a one-dimensional `Series`

### The Bad Way

- Suppose you would like to track data about states from two different years
- You could represent this data simply using Python tuples as keys

In [ ]:
index = [
    ("California", 2010),
    ("California", 2020),
    ("New York", 2010),
    ("New York", 2020),
    ("Texas", 2010),
    ("Texas", 2020),
]
populations = [37253956, 39538223, 19378102, 20201249, 25145561, 29145505]
pop = pd.Series(populations, index=index)
pop

- With this indexing scheme, you can straightforwardly index or slice the series based on this tuple index

In [ ]:
pop.loc[("California", 2020):("Texas", 2010)]

- But the convenience ends there
- For example, if you need to select all values from 2010, you'll need to do some messy (and potentially slow) munging to make it happen

In [ ]:
pop.loc[[i for i in pop.index if i[1] == 2010]]

### The Better Way: The Pandas MultiIndex

- Pandas provides a better way
- Our tuple-based indexing is essentially a rudimentary multi-index
- The Pandas `MultiIndex` type gives us the types of operations we wish to have
- We can create a multi-index from the tuples as follows

In [ ]:
index = pd.MultiIndex.from_tuples(index, names=["state", "year"])
index

- The `MultiIndex` represents multiple *levels* of indexing (the state name and years in this case) as well as multiple *labels* for each data point which encode these levels
- If we reindex our series with this `MultiIndex`, we see the hierarchical representation of the data

In [ ]:
pop = pop.reindex(index)
pop

- Here the first two columns of the `Series` representation show the multiple index values, while the third column shows the data
- In this multi-index representation, any blank entry indicates the same value as the line above it

- Now to access all data for which the second index is 2020, we can use the Pandas slicing notation

In [ ]:
pop.loc[:, 2020]

### MultiIndex as Extra Dimension

- We could easily have stored this two-dimensional data using a simple `DataFrame` with index and column labels
- In fact, the `unstack` method will quickly convert a multiply indexed `Series` into a conventionally indexed `DataFrame`

In [ ]:
pop_df = pop.unstack()
pop_df

- Naturally, the ``stack`` method provides the opposite operation

In [ ]:
pop_df.stack()

- However, a multi-index allows us to to manipulate data of arbitrary dimensions in a `Series` or `DataFrame`
- Each extra level in a multi-index represents an extra dimension of data 
- Concretely, we might want to add another column of demographic data for each state at each year (say, population under 18)
- With a `MultiIndex` this is as easy as adding another column to the ``DataFrame``

In [ ]:
pop_df = pd.DataFrame({"total": pop, "under18": [9284094, 8898092, 4318033, 4181528, 6879014, 7432474]})
pop_df

- In addition, all the ufuncs and other functionality work with hierarchical indices as well
- Here we compute the fraction of people under 18 by year, given the above data

In [ ]:
f_u18 = pop_df.loc[:, "under18"] / pop_df.loc[:, "total"]
f_u18.unstack()

## Methods of MultiIndex Creation

- The most straightforward way to construct a multiply indexed `Series` or `DataFrame` is to simply pass a list of two or more index arrays to the constructor

In [ ]:
df = pd.DataFrame(
    np.random.rand(4, 2),
    index=[["a", "a", "b", "b"], [1, 2, 1, 2]],
    columns=["data1", "data2"],
)
df

- The work of creating the ``MultiIndex`` is done in the background
- Similarly, if you pass a dictionary with appropriate tuples as keys, Pandas will automatically recognize this and use a ``MultiIndex`` by default

In [ ]:
data = {
    ("California", 2010): 37253956,
    ("California", 2020): 39538223,
    ("New York", 2010): 19378102,
    ("New York", 2020): 20201249,
    ("Texas", 2010): 25145561,
    ("Texas", 2020): 29145505,
}
pd.Series(data)

### Explicit MultiIndex Constructors

- For more flexibility in how the index is constructed, you can instead use the constructor methods available in the `pd.MultiIndex` class

- For example, you can construct a `MultiIndex` from a simple list of arrays giving the index values within each level

In [ ]:
pd.MultiIndex.from_arrays([["a", "a", "b", "b"], [1, 2, 1, 2]])

- Or you can construct it from a Cartesian product of single indices

In [ ]:
pd.MultiIndex.from_product([["a", "b"], [1, 2]])

- Similarly, you can construct a `MultiIndex` directly using its internal encoding by passing `levels` (a list of lists containing available index values for each level) and `codes` (a list of lists that reference these labels)

In [ ]:
pd.MultiIndex(levels=[["a", "b"], [1, 2]], codes=[[0, 0, 1, 1], [0, 1, 0, 1]])

### Indexing and Slicing a MultiIndex

- Indexing and slicing on a `MultiIndex` is designed to be intuitive, and it helps if you think about the indices as added dimensions

In [ ]:
pop

- We can access single elements by indexing with multiple terms

In [ ]:
pop.loc["California", 2010]

- Note that `pop` is a `Series`
- Since we have a 2-dimensional `MultiIndex`, the indexing looks like we are indexing a `DataFrame` with 1-dimensional row and column `Index`es

- The `MultiIndex` also supports *partial indexing*, or indexing just one of the levels in the index
- The result is another `Series`, with the lower-level indices maintained

In [ ]:
pop.loc["California"]

- Partial slicing is available as well, as long as the `MultiIndex` is sorted

In [ ]:
pop.loc["California":"New York"]

- With sorted indices, partial indexing can be performed on lower levels by passing an empty slice in the first index

In [ ]:
pop.loc[:, 2010]

- Other types of indexing and selection work as well; for example, selection based on Boolean masks

In [ ]:
pop.loc[pop > 22_000_000]

- Selection based on fancy indexing also works

In [ ]:
pop.loc[["California", "Texas"]]

### Exercises

Given the multi-indexed `Series` named `pop` from above

1. Select the population information for New York
2. Select the population information for all states from 2020
3. Select the population information for Texas from 2010 (bonus: retain the index)

In [ ]:
# Solve the exercises here

In [ ]:
pop.loc["New York"]

In [ ]:
pop.loc[:, 2020]

In [ ]:
pop.loc[[("Texas", 2010)]] # fancy indexing with a tuple preserves the index

In [ ]:
# Refresher

df = pd.DataFrame(np.random.randn(3, 3), index=[13, 4, -8], columns=["a", "b", "c"])
df

# we can use .iloc to access the implicit index
# df.iloc[0] # first row
# df.iloc[1:] # second row to end
# df.iloc[:, [2]] # all rows second third column (with fancy indexing)

# we can use .loc to access the explicit index
# df.loc[13, "a"]
df.loc[[4, -8], ["b", "a"]]

In [ ]:
# Multi-index

ser = pd.Series({("a", 1, "f"): 2.3, ("a", 3, "g"): 5.4, ("b", 1, "f"): -1.2, ("b", 2, "f"): 3})

# we can use a multi-index to represent (and access) multi-dimensional data in pandas Series

display(ser)

# ser.loc["a", 3, "g"] # "a" in first level, 3 in second level, "g" in third level
ser.loc[:, 1] # all in first level, 1 in second level

### MultiIndex for DataFrames (Columns)

- In a `DataFrame` both the rows and columns can have multiple levels
- Consider the following, which is a mock-up of some (somewhat realistic) medical data

In [ ]:
index = pd.MultiIndex.from_product([[2013, 2014], [1, 2]], names=["year", "visit"])
columns = pd.MultiIndex.from_product([["Bob", "Guido", "Sue"], ["HR", "Temp"]], names=["subject", "type"])

data = np.round(np.random.randn(4, 6), 1)
data[:, ::2] *= 10
data += 37

health_data = pd.DataFrame(data, index=index, columns=columns)
health_data

- This is fundamentally four-dimensional data, where the dimensions are the subject, the measurement type, the year, and the visit number
- With this in place we can, for example, index the top-level column by the person's name and get a full `DataFrame` containing just that person's information

In [ ]:
health_data.loc[:, "Guido"]
# health_data.loc[:, ["Guido"]] # use fancy indexing to retain the index

- When accessing multiple levels we cannot follow the syntax using for a `Series`
- Different levels in multi-indexed `Series` are accessed using comma separated labels

In [ ]:
ser = pd.Series(range(8), index=pd.MultiIndex.from_product([["a", "b"], ["c", "d"], ["e", "f"]]))
ser

In [ ]:
ser.loc["a", "c", "e"]

- In a `DataFrame` comma separation is already used to distinguish between row indexing and column indexing
- To access additional levels in a `DataFrame` we can use tuples
- For example, we can use a tuple of a name and a measurement type to access a specific person's measurements

In [ ]:
health_data.loc[:, ("Guido", "HR")]
# health_data.loc[:, [('Guido', 'HR')]] # use fancy indexing to retain the index

- However, working with slices within these index tuples is not especially convenient
- Trying to create a slice within a tuple will lead to a syntax error

In [ ]:
health_data.loc[(:, 1), (:, 'HR')]

- You can get around this by using an `IndexSlice` object

In [ ]:
idx = pd.IndexSlice
health_data.loc[idx[:, 1], idx[:, "HR"]]

- Note that when using `IndexSlice` both the row and column labels need to be filled
- That is, the shorthand notation for row indexing of `df.loc[idx[:, 1]]` will not work

In [ ]:
# health_data.loc[idx[:, 1]] # fails
health_data.loc[idx[:, 1], :] # we must provide an empty column slice

### Exercises

Given the multi-indexed DataFrame from above

1. Select the health information for the year 2013
2. Select the health information for Sue
3. Select Sue's temperature values for her first visit in 2014
4. Select the temperature for all second visits

In [ ]:
# Solve the exercises here

In [ ]:
health_data.loc[2013]

In [ ]:
health_data.loc[:, "Sue"]

In [ ]:
health_data.loc[(2014, 1), ("Sue", "Temp")]

In [ ]:
health_data.loc[idx[:, 2], idx[:, "Temp"]]

## Rearranging Multi-Indexes

- There are a number of operations that will preserve all the information in the dataset, but rearrange it for the purposes of various computations

### Stacking and Unstacking Indices

- As we saw briefly before, it is possible to convert a dataset from a stacked multi-index to a simple two-dimensional representation
- We can optionally specifying the level to unstack

In [ ]:
pop.unstack(level=1) # by default level=-1 meaning the final level

In [ ]:
pop.unstack(level=0)

- The opposite of `unstack` is `stack`, which here can be used to recover the original series

In [ ]:
pop.unstack().stack()

### Index Setting and Resetting

- Another way to rearrange hierarchical data is to turn the index labels into columns; this can be accomplished with the `reset_index` method
- Calling this on the population `Series` will result in a `DataFrame` with `state` and `year` columns holding the information that was formerly in the index
- For clarity, we can optionally specify the name of the data for the column representation

In [ ]:
pop_flat = pop.reset_index(name="population")
pop_flat

- A common pattern is to build a `MultiIndex` from the column values
- This can be done with the `set_index` method of the `DataFrame`, which returns a multiply indexed `DataFrame`

In [ ]:
pop_flat.set_index(["state", "year"])

### Exercises

1. Given the `health_data` `DataFrame` from above, move the `type` level into the row index and the `year` and `visit` levels into the column index to allow easily selecting all the HR or temperature data for a subject. The final `DataFrame` should look like this:

<table border="1" class="dataframe">
  <thead>
    <tr>
      <th>subject</th>
      <th colspan="4" halign="left">Bob</th>
      <th colspan="4" halign="left">Guido</th>
      <th colspan="4" halign="left">Sue</th>
    </tr>
    <tr>
      <th>year</th>
      <th colspan="2" halign="left">2013</th>
      <th colspan="2" halign="left">2014</th>
      <th colspan="2" halign="left">2013</th>
      <th colspan="2" halign="left">2014</th>
      <th colspan="2" halign="left">2013</th>
      <th colspan="2" halign="left">2014</th>
    </tr>
    <tr>
      <th>visit</th>
      <th>1</th>
      <th>2</th>
      <th>1</th>
      <th>2</th>
      <th>1</th>
      <th>2</th>
      <th>1</th>
      <th>2</th>
      <th>1</th>
      <th>2</th>
      <th>1</th>
      <th>2</th>
    </tr>
    <tr>
      <th>type</th>
      <th></th>
      <th></th>
      <th></th>
      <th></th>
      <th></th>
      <th></th>
      <th></th>
      <th></th>
      <th></th>
      <th></th>
      <th></th>
      <th></th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>HR</th>
      <td>52.0</td>
      <td>37.0</td>
      <td>24.0</td>
      <td>30.0</td>
      <td>47.0</td>
      <td>45.0</td>
      <td>12.0</td>
      <td>32.0</td>
      <td>47.0</td>
      <td>53.0</td>
      <td>44.0</td>
      <td>41.0</td>
    </tr>
    <tr>
      <th>Temp</th>
      <td>37.5</td>
      <td>37.2</td>
      <td>37.8</td>
      <td>37.9</td>
      <td>36.9</td>
      <td>37.7</td>
      <td>35.0</td>
      <td>37.8</td>
      <td>36.1</td>
      <td>35.2</td>
      <td>36.2</td>
      <td>39.0</td>
    </tr>
  </tbody>
</table>

In [ ]:
# Solve the exercise here

In [ ]:
health_data.stack().unstack(0).unstack(0)